# 📓 GOOGLE COLAB NOTEBOOK

## JSON → Corporate PPT Generator

## 🟦 CELL 1 — Install Dependencies

In [21]:
!pip install python-pptx pydantic

## 🟦 CELL 2 — Imports & Constants

In [22]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pathlib import Path
import json

## 🟦 CELL 3 — Theme Definition (LOCKED)

> Single theme used for **all slides**

In [23]:
THEME = {
    "background": RGBColor(245, 245, 245),
    "title_color": RGBColor(47, 93, 140),
    "text_color": RGBColor(31, 41, 51),
    "muted_text": RGBColor(95, 108, 114),
    "table_header_bg": RGBColor(238, 242, 246),
    "border": RGBColor(217, 221, 225),

    "status": {
        "completed": RGBColor(76, 175, 80),
        "in-progress": RGBColor(74, 144, 226),
        "pending": RGBColor(224, 168, 0)
    },

    "font": {
        "family": "Calibri",
        "title": 32,
        "section": 22,
        "body": 14,
        "small": 11
    }
}

## 🟦 CELL 4 — PPT Helper Functions

In [24]:
def add_title(slide, text):
    # Create a title text box
    title_shape = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(12.33), Inches(1))
    tf = title_shape.text_frame
    p = tf.paragraphs[0]
    p.text = text
    p.font.size = Pt(THEME["font"]["title"])
    p.font.bold = True
    p.font.color.rgb = THEME["title_color"]
    
    # Add a decorative underline
    line = slide.shapes.add_shape(
        MSO_SHAPE.RECTANGLE, Inches(0.5), Inches(1.1), Inches(12.33), Inches(0.03)
    )
    line.fill.solid()
    line.fill.fore_color.rgb = THEME["title_color"]
    line.line.fill.background()

def add_text(slide, text, top=1.5):
    box = slide.shapes.add_textbox(Inches(0.5), Inches(top), Inches(12.33), Inches(2))
    tf = box.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.text = text
    p.font.size = Pt(THEME["font"]["body"])
    p.font.color.rgb = THEME["text_color"]

def add_banner(slide, text, top=1.5):
    box = slide.shapes.add_shape(
        MSO_SHAPE.ROUNDED_RECTANGLE,
        Inches(0.5), Inches(top), Inches(12.33), Inches(1.2)
    )
    box.fill.solid()
    box.fill.fore_color.rgb = RGBColor(235, 242, 250) # Light blue bg
    box.line.color.rgb = THEME["title_color"]
    
    tf = box.text_frame
    tf.margin_left = Inches(0.2)
    p = tf.paragraphs[0]
    p.text = text
    p.font.size = Pt(16)
    p.font.color.rgb = THEME["text_color"]
    p.alignment = PP_ALIGN.CENTER

## 🟦 CELL 5 — Table Renderer

In [25]:
def add_table(slide, table_json, top=2.0):
    rows = len(table_json["rows"]) + 1
    cols = len(table_json["columns"])
    
    # Wider table for 16:9
    table_width = Inches(12.33)
    col_width = table_width / cols
    
    shape = slide.shapes.add_table(
        rows, cols,
        Inches(0.5), Inches(top),
        table_width, Inches(0.5 * rows)
    )
    table = shape.table

    # Set column widths
    for i in range(cols):
        table.columns[i].width = int(col_width)

    # Header
    for i, col in enumerate(table_json["columns"]):
        cell = table.cell(0, i)
        cell.text = col["label"]
        cell.fill.solid()
        cell.fill.fore_color.rgb = THEME["table_header_bg"]
        p = cell.text_frame.paragraphs[0]
        p.font.bold = True
        p.font.size = Pt(14)
        p.font.color.rgb = THEME["title_color"]

    # Rows
    for r, row in enumerate(table_json["rows"], start=1):
        for c, col in enumerate(table_json["columns"]):
            val = row[col["key"]]
            cell = table.cell(r, c)
            cell.text = val
            p = cell.text_frame.paragraphs[0]
            p.font.size = Pt(12)
            p.font.color.rgb = THEME["text_color"]
            
            # Optional: Color code status
            if col["key"] == "status":
                status_key = val.lower().replace(" ", "-")
                if status_key in THEME["status"]:
                    p.font.color.rgb = THEME["status"][status_key]
                    p.font.bold = True

## 🟦 CELL 6 — Card Renderer

In [26]:
def add_card(slide, title, items, left, top, width=6.0, height=3.5):
    # Card Container
    box = slide.shapes.add_shape(
        MSO_SHAPE.ROUNDED_RECTANGLE,
        Inches(left), Inches(top), Inches(width), Inches(height)
    )
    box.fill.solid()
    box.fill.fore_color.rgb = RGBColor(255, 255, 255)
    box.line.color.rgb = THEME["border"]
    box.line.width = Pt(1.5)
    
    # Shadow effect (simulated with a darker box behind? No, too complex. Just clean border)

    tf = box.text_frame
    tf.margin_top = Inches(0.2)
    tf.margin_left = Inches(0.2)
    tf.margin_right = Inches(0.2)

    # Title
    p = tf.paragraphs[0]
    p.text = title
    p.font.size = Pt(18)
    p.font.bold = True
    p.font.color.rgb = THEME["title_color"]
    p.alignment = PP_ALIGN.CENTER
    
    # Spacer
    tf.add_paragraph()

    # Items
    for item in items:
        p = tf.add_paragraph()
        p.text = f"• {item}"
        p.font.size = Pt(14)
        p.font.color.rgb = THEME["text_color"]
        p.space_before = Pt(5)

## 🟦 CELL 7 — Timeline Renderer

In [27]:
def add_timeline(slide, timeline):
    base_y = 3.5
    start_x = 1.0
    # Scale to fit 16:9 width (approx 11 inches usable)
    # Find max end
    max_end = max(p["end"] for p in timeline["phases"])
    scale_x = 11.0 / max_end if max_end > 0 else 1
    
    # Draw axis
    axis = slide.shapes.add_shape(
        MSO_SHAPE.RECTANGLE, 
        Inches(start_x), Inches(base_y + 0.5), Inches(max_end * scale_x), Inches(0.05)
    )
    axis.fill.solid()
    axis.fill.fore_color.rgb = THEME["border"]

    for i, phase in enumerate(timeline["phases"]):
        x = start_x + phase["start"] * scale_x
        w = (phase["end"] - phase["start"]) * scale_x
        
        # Alternate heights to avoid overlap
        y_offset = 0 if i % 2 == 0 else 1.2
        
        # Phase Box
        box = slide.shapes.add_shape(
            MSO_SHAPE.ROUNDED_RECTANGLE,
            Inches(x), Inches(base_y + y_offset), Inches(w), Inches(0.8)
        )
        
        # Color cycle
        colors = [THEME["status"]["completed"], THEME["status"]["in-progress"], THEME["title_color"]]
        color = colors[i % len(colors)]
        
        box.fill.solid()
        box.fill.fore_color.rgb = color
        box.line.fill.background()
        
        tf = box.text_frame
        p = tf.paragraphs[0]
        p.text = phase["name"]
        p.font.color.rgb = RGBColor(255, 255, 255)
        p.font.bold = True
        p.alignment = PP_ALIGN.CENTER

## 🟦 CELL 8 — FULL JSON INPUT (Random Project Proposal)

> **ONLY this JSON controls the PPT**

In [28]:
ppt_json = {
  "meta": {
    "title": "Retail Billing Platform Modernization",
    "author": "Ashish",
    "date": "Dec 2025"
  },
  "slides": [

    {
      "type": "cover",
      "title": "Retail Billing Platform Modernization",
      "subtitle": "Project Proposal"
    },

    {
      "type": "content",
      "title": "Project Overview",
      "banner": "This proposal outlines scope, timeline and execution approach.",
      "text": "The goal is to modernize the billing platform for scalability and compliance."
    },

    {
      "type": "table",
      "title": "Project Timeline & Status",
      "table": {
        "columns": [
          {"key": "phase", "label": "Phase"},
          {"key": "timeline", "label": "Timeline"},
          {"key": "status", "label": "Status"}
        ],
        "rows": [
          {"phase": "Design", "timeline": "Week 1–2", "status": "Completed"},
          {"phase": "Development", "timeline": "Week 3–6", "status": "Completed"},
          {"phase": "UAT", "timeline": "Week 7", "status": "In Progress"}
        ]
      }
    },

    {
      "type": "cards",
      "title": "Activities & Risks",
      "cards": [
        {
          "title": "Completed",
          "items": ["UI Design", "Core APIs", "Admin Portal"]
        },
        {
          "title": "Risks",
          "items": ["Stakeholder approval", "UAT feedback"]
        }
      ]
    },

    {
      "type": "timeline",
      "title": "High Level Timeline",
      "timeline": {
        "phases": [
          {"name": "Design", "start": 0, "end": 2},
          {"name": "Build", "start": 2, "end": 6},
          {"name": "UAT", "start": 6, "end": 7}
        ]
      }
    }
  ]
}

## 🟦 CELL 9 — PPT Generator (JSON → PPT)

In [29]:
prs = Presentation()
# Set 16:9 Aspect Ratio (Widescreen)
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)

for slide_json in ppt_json["slides"]:
    # Use blank layout
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    
    # Add background color (optional, but clean white is standard)
    background = slide.background
    fill = background.fill
    fill.solid()
    fill.fore_color.rgb = THEME["background"]

    add_title(slide, slide_json["title"])

    if slide_json["type"] == "content":
        add_banner(slide, slide_json["banner"], top=1.8)
        add_text(slide, slide_json["text"], top=3.5)

    if slide_json["type"] == "table":
        add_table(slide, slide_json["table"], top=2.0)

    if slide_json["type"] == "cards":
        # Two cards side by side
        add_card(slide, slide_json["cards"][0]["title"],
                 slide_json["cards"][0]["items"], left=0.5, top=2.0, width=6.0, height=4.0)
        add_card(slide, slide_json["cards"][1]["title"],
                 slide_json["cards"][1]["items"], left=6.8, top=2.0, width=6.0, height=4.0)

    if slide_json["type"] == "timeline":
        add_timeline(slide, slide_json["timeline"])
        
    # Footer
    footer = slide.shapes.add_textbox(Inches(0.5), Inches(7.0), Inches(12.33), Inches(0.5))
    p = footer.text_frame.paragraphs[0]
    p.text = f"{ppt_json['meta']['title']} | {ppt_json['meta']['date']}"
    p.font.size = Pt(10)
    p.font.color.rgb = THEME["muted_text"]
    p.alignment = PP_ALIGN.RIGHT

## 🟦 CELL 10 — Save & Download PPT

In [30]:
import base64
import os
from IPython.display import HTML, display

output_path = "project_proposal.pptx"
prs.save(output_path)
print(f"Presentation saved to: {output_path}")

if os.path.exists(output_path):
    with open(output_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    
    # Create a clickable download link
    download_link = f'<a href="data:application/vnd.openxmlformats-officedocument.presentationml.presentation;base64,{b64}" download="{output_path}" style="font-size: 20px; font-weight: bold; color: blue;">⬇️ Click Here to Download PPT</a>'
    display(HTML(download_link))

Presentation saved to: project_proposal.pptx
